In [1]:
import numpy as np
import pandas as pd
import csv
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

In [2]:
weather_data = pd.read_csv("data/A榜-训练集_分布式光伏发电预测_气象变量数据.csv",encoding = 'gbk')
weather_data = weather_data[['光伏用户编号','时间','辐照强度（J/m2）','温度（K）']]

In [3]:
from datetime import datetime, timedelta
power_data = pd.read_csv("data\A榜-训练集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')

df = pd.DataFrame(power_data)

# 创建以15分钟为间隔的数据
new_data = []
start_date = datetime(2022, 1, 3, 0, 0)  # 起始时间为2022-01-03 00:00
for i in range(df.shape[0]):
    row = df.iloc[i]
    for j in range(1, 97):
        time = start_date + timedelta(minutes=15*(j-1))
        new_data.append([row['光伏用户编号'], time])

# Convert '时间' column to datetime64 data type
weather_data['时间'] = pd.to_datetime(weather_data['时间'])
power_data['时间'] = pd.to_datetime(power_data['时间'])

power_data_drop = pd.DataFrame(new_data, columns=['光伏用户编号', '时间'])
train_data = pd.merge(power_data_drop, weather_data, on=['光伏用户编号', '时间'],how='left')
train_data['光伏用户编号'].replace({'f1': '1', 'f2': '2', 'f3': '3', 'f4': '4', 'f5': '5',
                   'f6': '6', 'f7': '7', 'f8': '8', 'f9': '9'}, inplace=True)
train_data

,光伏用户编号,时间,辐照强度（J/m2）,温度（K）
0,1,2022-01-03 00:00:00,0.0,280.6320
1,1,2022-01-03 00:15:00,0.0,280.6308
2,1,2022-01-03 00:30:00,0.0,280.5719
3,1,2022-01-03 00:45:00,0.0,280.4776
4,1,2022-01-03 01:00:00,0.0,280.3705
...,...,...,...,...
416347,9,2022-01-03 22:45:00,0.0,285.0616
416348,9,2022-01-03 23:00:00,0.0,285.0117
416349,9,2022-01-03 23:15:00,0.0,284.9275
416350,9,2022-01-03 23:30:00,0.0,284.8174


In [4]:
from sklearn.preprocessing import MinMaxScaler
n_tasks = 96
# 特征工程
train_data['时间'] = pd.to_datetime(train_data['时间'])
train_data['月份'] = train_data['时间'].dt.month
train_data['日期'] = train_data['时间'].dt.day
train_data['小时'] = train_data['时间'].dt.hour
train_data['分钟'] = train_data['时间'].dt.minute

# 按时间排序数据
X_train = train_data.sort_values(by='时间')

X_train = pd.DataFrame(X_train)

# 选择特征和目标变量
X_train = X_train.drop('时间', axis=1)

# 标准化或归一化特征
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

n_features = X_train.shape[1]
X_train

array([[0.        , 0.        , 0.16661925, ..., 0.        , 0.        ,
        0.        ],
       [0.125     , 0.        , 0.41880367, ..., 0.        , 0.        ,
        0.        ],
       [0.875     , 0.        , 0.30994583, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.25      , 0.        , 0.34652787, ..., 0.        , 1.        ,
        1.        ],
       [0.25      , 0.        , 0.34652787, ..., 0.        , 1.        ,
        1.        ],
       [1.        , 0.        , 0.4025824 , ..., 0.        , 1.        ,
        1.        ]])

In [5]:
X_train = np.ravel(X_train)
len(X_train)
# X_train 转换为(n_samples, n_tasks, n_features)
n_tasks = 96  # 假设有2个任务
n_features = 7

# 将 X_train 重塑为 (n_samples, n_tasks, n_features) 的形状
X_train = X_train.reshape(-1, n_tasks, n_features)
print(X_train.shape)

(4337, 96, 7)


In [6]:
y_train = power_data[['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27', 'p28', 'p29', 'p30', 'p31', 'p32', 'p33', 'p34', 'p35', 'p36', 'p37', 'p38', 'p39', 'p40', 'p41', 'p42', 'p43', 'p44', 'p45', 'p46', 'p47', 'p48', 'p49', 'p50', 'p51', 'p52', 'p53', 'p54', 'p55', 'p56', 'p57', 'p58', 'p59', 'p60', 'p61', 'p62', 'p63', 'p64', 'p65', 'p66', 'p67', 'p68', 'p69', 'p70', 'p71', 'p72', 'p73', 'p74', 'p75', 'p76', 'p77', 'p78', 'p79', 'p80', 'p81', 'p82', 'p83', 'p84', 'p85', 'p86', 'p87', 'p88', 'p89', 'p90', 'p91', 'p92', 'p93', 'p94', 'p95', 'p96']]
n_tasks = 96
y_train = np.ravel(y_train)
# 调整y_train的形状为(n_samples, n_tasks)
y_train = y_train.reshape(-1, n_tasks)

y_train.shape

(4337, 96)

In [7]:
weather_test = pd.read_csv("data/A榜-测试集_分布式光伏发电预测_气象变量数据.csv",encoding = 'gbk')
weather_test = weather_test[['光伏用户编号','时间','辐照强度（J/m2）','温度（K）']]
power_test = pd.read_csv("data\A榜-测试集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')

In [8]:
df = pd.DataFrame(power_test)

# 创建新的DataFrame,将p1到p96作为行索引
new_data = []
start_date = datetime(2023, 5, 1, 0, 0)
for i in range(df.shape[0]):
    row = df.iloc[i]
    for j in range(1, 97):
        time = start_date + timedelta(minutes=15*(j-1))
        new_data.append([row['光伏用户编号'], time])

power_test = pd.DataFrame(new_data, columns=['光伏用户编号', '时间'])
power_test['光伏用户编号'].replace({'f1': '1', 'f2': '2', 'f3': '3', 'f4': '4', 'f5': '5',
                   'f6': '6', 'f7': '7', 'f8': '8', 'f9': '9'}, inplace=True)
power_test

,光伏用户编号,时间
0,1,2023-05-01 00:00:00
1,1,2023-05-01 00:15:00
2,1,2023-05-01 00:30:00
3,1,2023-05-01 00:45:00
4,1,2023-05-01 01:00:00
...,...,...
79195,9,2023-05-01 22:45:00
79196,9,2023-05-01 23:00:00
79197,9,2023-05-01 23:15:00
79198,9,2023-05-01 23:30:00


In [9]:
# Convert '时间' column to datetime64 data type
weather_test['时间'] = pd.to_datetime(weather_test['时间'])
power_test['时间'] = pd.to_datetime(power_test['时间'])
# Now merge the DataFrames
test_data = pd.merge(power_test,weather_test, on=['光伏用户编号','时间'],how='left')
# 特征工程
test_data['时间'] = pd.to_datetime(test_data['时间'])
test_data['月份'] = test_data['时间'].dt.month
test_data['日期'] = test_data['时间'].dt.day
test_data['小时'] = test_data['时间'].dt.hour
test_data['分钟'] = test_data['时间'].dt.minute
test_data = test_data.drop('时间', axis=1)
test_data['辐照强度（J/m2）'] = weather_test['辐照强度（J/m2）']
test_data['温度（K）'] = weather_test['温度（K）']
test_data

,光伏用户编号,辐照强度（J/m2）,温度（K）,月份,日期,小时,分钟
0,1,0.0,287.6853,5,1,0,0
1,1,0.0,287.7061,5,1,0,15
2,1,0.0,287.6017,5,1,0,30
3,1,0.0,287.4312,5,1,0,45
4,1,0.0,287.2534,5,1,1,0
...,...,...,...,...,...,...,...
79195,9,0.0,298.5737,5,1,22,45
79196,9,0.0,298.5271,5,1,23,0
79197,9,0.0,298.6875,5,1,23,15
79198,9,0.0,298.9674,5,1,23,30


In [10]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder

X_test = pd.DataFrame(test_data)

# 标准化或归一化特征
scaler = MinMaxScaler()
X_test = scaler.fit_transform(X_test)
X_test = np.ravel(X_test)
X_test = X_test.reshape(-1, n_tasks, n_features)

X_test.shape

(825, 96, 7)

In [11]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.impute import SimpleImputer
from tqdm import tqdm

# 处理y_train中的NaN值
imputer = SimpleImputer(strategy='mean')  # 使用平均值填充NaN

y_train = imputer.fit_transform(y_train)

# 定义核函数
rbf_kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2))

# 创建GPR模型
gpr = GaussianProcessRegressor(kernel=rbf_kernel, n_restarts_optimizer=10, alpha=1e-6)

# 使用稀疏近似
gpr.set_params(optimizer='fmin_l_bfgs_b', copy_X_train=True)

# 使用核矩阵近似
gpr.set_params(normalize_y=True)

# 训练模型
print("Training GPR model...")
gpr.fit(X_train.reshape(X_train.shape[0], -1), y_train)

Training GPR model...
Predicting on test set...


100%|█████████████████████████████████████████████████████████████████████████████████| 7/7 [1:17:37<00:00, 665.42s/it]


In [21]:
# 在测试集上预测
print("Predicting on test set...")
y_pred = np.zeros((X_test.shape[0], 96))
for i in tqdm(range(96)):
    gpr.kernel_.theta = rbf_kernel.theta  # 重置核函数参数
    y_train_i = y_train[:, i].reshape(-1, 1)
    y_pred[:, i] = gpr.predict(X_test.reshape(X_test.shape[0], -1)).ravel()

Predicting on test set...


100%|██████████████████████████████████████████████████████████████████████████████████| 96/96 [01:26<00:00,  1.11it/s]


In [22]:
power_test = pd.read_csv("data\A榜-测试集_分布式光伏发电预测_实际功率数据.csv",encoding = 'gbk')
power_test

,光伏用户编号,综合倍率,时间
0,f1,80,2023-5-1 0:00
1,f1,80,2023-5-2 0:00
2,f1,80,2023-5-3 0:00
3,f1,80,2023-5-4 0:00
4,f1,80,2023-5-5 0:00
...,...,...,...
820,f9,8000,2023-7-27 0:00
821,f9,8000,2023-7-30 0:00
822,f9,8000,2023-7-31 0:00
823,f9,8000,2023-5-15 0:00


In [23]:
# 重构预测结果
#y_pred = y_pred.reshape(-1, 96)
print(len(y_pred))
y_pred_df = pd.DataFrame(y_pred, columns=['p'+str(i) for i in range(1,97)])

# 在 power_test 中创建 p1 到 p96 列
for i in range(1, 97):
    power_test[f'p{i}'] = 0

# 将 y_pred_df 中的值拼接到 power_test 中对应的列
for col in y_pred_df.columns:
    power_test[col] = y_pred_df[col].values
power_test

825


,光伏用户编号,综合倍率,时间,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24,p25,p26,p27,p28,p29,p30,p31,p32,p33,p34,p35,p36,p37,p38,p39,p40,p41,p42,p43,p44,p45,p46,p47,p48,p49,p50,p51,p52,p53,p54,p55,p56,p57,p58,p59,p60,p61,p62,p63,p64,p65,p66,p67,p68,p69,p70,p71,p72,p73,p74,p75,p76,p77,p78,p79,p80,p81,p82,p83,p84,p85,p86,p87,p88,p89,p90,p91,p92,p93,p94,p95,p96
0,f1,80,2023-5-1 0:00,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137,-0.030137
1,f1,80,2023-5-2 0:00,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809,-0.032809
2,f1,80,2023-5-3 0:00,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420,-0.033420
3,f1,80,2023-5-4 0:00,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-0.034001,-

In [24]:

power_test = power_test.round(4)

# 将结果写入CSV文件,编码格式为UTF-8
power_test.to_csv('xxx.csv', encoding='utf-8', index=False)

In [25]:
pd.read_csv("xxx.csv",encoding='utf-8')

,光伏用户编号,综合倍率,时间,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24,p25,p26,p27,p28,p29,p30,p31,p32,p33,p34,p35,p36,p37,p38,p39,p40,p41,p42,p43,p44,p45,p46,p47,p48,p49,p50,p51,p52,p53,p54,p55,p56,p57,p58,p59,p60,p61,p62,p63,p64,p65,p66,p67,p68,p69,p70,p71,p72,p73,p74,p75,p76,p77,p78,p79,p80,p81,p82,p83,p84,p85,p86,p87,p88,p89,p90,p91,p92,p93,p94,p95,p96
0,f1,80,2023-5-1 0:00,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301,-0.0301
1,f1,80,2023-5-2 0:00,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328,-0.0328
2,f1,80,2023-5-3 0:00,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334,-0.0334
3,f1,80,2023-5-4 0:00,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340,-0.0340
4,f1,80,2023-5-5 0:00,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.0343,-0.